# H-NEG on Kaggle: step-by-step run (Gemma 4 E2B / E4B GGUF on a T4)

**Code:** [GitHub repository](https://github.com/PDawda/hneg-gemma4-agent) · **License:** Apache 2.0

Run the cells **in order**. Each step prints what success looks like.

**Notebook settings (right sidebar) before starting:**
- *Accelerator*: **GPU T4 x2**
- *Internet*: **On** (requires a phone-verified Kaggle account)
- *Persistence*: optional; files under `/kaggle/working` are saved with each version

| Step | What | Time (approx.) |
|---|---|---|
| 1 | Environment check | seconds |
| 2 | Get the code | seconds |
| 3 | Python dependencies | 1 min |
| 4 | llama.cpp with CUDA (cached after first run) | 15-30 min first time, seconds after |
| 5 | Unit tests | 30 s |
| 6 | Sandbox checks on Gemma 4 E2B | 5-10 min (incl. 3.1 GB download) |
| 7 | Build validated SWE-bench pilot tasks | 10-40 min |
| 8 | Ablation benchmark, all bugs (resumable) | ~2.5 h on E4B for 20 bugs |
| 9 | Results, claim verification, calibration | seconds |

## Step 1 — Environment check
Expect: a Tesla T4 listed, Python 3.10+, and several GB free in `/kaggle/working` and `/kaggle/tmp`.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python --version
!df -h /kaggle/working /kaggle/tmp | sed 1d
!curl -sI https://huggingface.co | head -1   # expect HTTP/2 200 (Internet on)

## Step 2 — Get the code
Pick **one** source:
- `"dataset"`: you uploaded `hneg-gemma4-agent.zip` as a Kaggle Dataset and attached it (*Add Input → Datasets*).
- `"github"`: the repo is on GitHub; set `GITHUB_URL`.

After uploading a **new version** of the Dataset: in the right sidebar click the dataset's ⋮ → *Check for updates*, then rerun this cell.

In [ ]:
import os, glob, shutil, subprocess, sys

CODE_SOURCE = "dataset"          # "dataset" or "github"
GITHUB_URL = "https://github.com/<you>/hneg-gemma4-agent.git"
REPO = "/kaggle/working/hneg-gemma4-agent"

# Always start from a fresh copy, so an updated Dataset version / new commit is picked up.
# (Results, traces, tasks and the llama.cpp wheel live outside REPO and are kept.)
os.chdir("/kaggle/working")
shutil.rmtree(REPO, ignore_errors=True)
if CODE_SOURCE == "github":
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO], check=True)
else:
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/hneg_config.yaml", recursive=True)]
    assert hits, "Attach the dataset containing hneg-gemma4-agent (Add Input -> Datasets)"
    shutil.copytree(hits[0], REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)
print("code at", os.getcwd()); print(sorted(os.listdir()))
print("experiments:", sorted(os.listdir("experiments")))

## Step 3 — Python dependencies
Kaggle already ships PyTorch with CUDA; we do **not** reinstall it.

In [ ]:
!pip install -q pyyaml tree-sitter tree-sitter-language-pack datasets huggingface_hub pytest
# Test-time dependencies of the SWE-bench repos used in the pilot (Step 7)
!pip install -q mpmath "flask<3" werkzeug requests

## Step 4 — llama.cpp with CUDA (`llama-cpp-python`)
Prebuilt CUDA wheels stop at 0.3.19, which is likely too old for Gemma 4, so the first run **builds from source** for the T4 (compute capability 7.5) and saves the wheel to `/kaggle/working/wheels`.

**After the first run:** create a Kaggle Dataset from `/kaggle/working/wheels` (e.g. named `llama-cpp-wheel`) and attach it. This cell then installs from it in seconds.

Expect: `GPU offload supported: True`.

Use the **T4**, not the P100: Kaggle's CUDA 13 toolkit no longer supports the P100 (Pascal) architecture.

In [ ]:
import glob, os, shutil, subprocess, sys, time
LLAMA_CPP_VERSION = "0.3.36"
LOG = "/kaggle/working/llama_build.log"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

# Locate the CUDA compiler: Kaggle has the toolkit under /usr/local/cuda, but not always on PATH
nvcc = shutil.which("nvcc") or next(iter(sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))), None)
print("python :", sys.version.split()[0])
print("nvcc   :", nvcc, "|", sh(f"{nvcc} --version | tail -1") if nvcc else "NOT FOUND")
print("cmake  :", sh("cmake --version | head -1") or "not installed (pip will fetch one)")
print("gcc    :", sh("gcc --version | head -1"))

cached = sorted(glob.glob("/kaggle/input/**/llama_cpp_python-*.whl", recursive=True))
t0 = time.time()
if cached:
    print("installing cached wheel:", cached[-1])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", cached[-1]], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diskcache"], check=True)
else:
    assert nvcc, "nvcc not found: is the accelerator set to GPU T4 x2?"
    os.makedirs("/kaggle/working/wheels", exist_ok=True)
    # Kaggle's broken /usr/local/bin/cmake shim: reinstall the pip cmake package
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake"], check=False)
    # CUDA 13 + CMake 3.31 may not find libcuda (target CUDA::cuda_driver). Point CMake at the
    # driver stub; without one, build with GGML_CUDA_NO_VMM=ON, which doesn't link libcuda.
    cuda_root = os.path.dirname(os.path.dirname(nvcc))
    libcuda = next(iter(sorted(glob.glob(f"{cuda_root}/lib64/stubs/libcuda.so")
                               + glob.glob(f"{cuda_root}/targets/*/lib/stubs/libcuda.so")
                               + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*"))), None)
    cmake_args = (f"-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_CUDA_COMPILER={nvcc} "
                  f"-DCUDAToolkit_ROOT={cuda_root} -DGGML_CCACHE=OFF")
    cmake_args += f" -DCUDA_cuda_driver_LIBRARY={libcuda}" if libcuda else " -DGGML_CUDA_NO_VMM=ON"
    print("libcuda:", libcuda or "not found -> GGML_CUDA_NO_VMM=ON")
    env = dict(os.environ, CMAKE_ARGS=cmake_args, CUDACXX=nvcc, FORCE_CMAKE="1",
               CMAKE_BUILD_PARALLEL_LEVEL=str(os.cpu_count() or 4),
               PATH=os.path.dirname(nvcc) + ":" + os.environ["PATH"],
               LIBRARY_PATH=":".join(filter(None, [os.path.dirname(libcuda) if libcuda else "",
                                                   os.environ.get("LIBRARY_PATH", "")])))
    print(f"building llama-cpp-python {LLAMA_CPP_VERSION} with CUDA (15-30 min); log: {LOG}")
    with open(LOG, "w") as fh:
        r = subprocess.run([sys.executable, "-m", "pip", "wheel", "-v", f"llama-cpp-python=={LLAMA_CPP_VERSION}",
                            "--no-deps", "-w", "/kaggle/working/wheels"], stdout=fh, stderr=subprocess.STDOUT, env=env)
    if r.returncode != 0:
        lines = open(LOG, errors="replace").read().splitlines()
        errs = [l for l in lines if any(k in l.lower() for k in ("error", "fatal", "not found", "failed"))]
        print("BUILD FAILED. Error lines:\n" + "\n".join(errs[-25:]))
        print("\n--- last 40 log lines ---\n" + "\n".join(lines[-40:]))
        raise SystemExit("llama-cpp-python build failed (see above)")
    whl = sorted(glob.glob("/kaggle/working/wheels/llama_cpp_python-*.whl"))[-1]
    # --no-deps: don't let pip upgrade Kaggle's numpy; add the one small runtime dep separately
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", whl], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diskcache"], check=True)
print(f"done in {time.time() - t0:.0f}s")

In [ ]:
import llama_cpp, numpy, torch, pandas
print("llama-cpp-python", llama_cpp.__version__)
print("GPU offload supported:", llama_cpp.llama_supports_gpu_offload())
print("numpy", numpy.__version__, "| torch", torch.__version__, "CUDA", torch.cuda.is_available(), "| pandas", pandas.__version__)
x = torch.from_numpy(numpy.arange(4)).cuda().sum().item()   # numpy <-> torch <-> GPU round trip
print("numpy/torch/GPU interop OK:", x == 6)

## Step 5 — Unit tests
Expect: `74 passed, 1 skipped`. The skipped one is the live-GGUF test; Step 6 covers it.


In [ ]:
!python -m pytest -q tests

## Step 6 — Sandbox checks on Gemma 4 E2B (GGUF)
Downloads `unsloth/gemma-4-E2B-it-GGUF / gemma-4-E2B-it-Q4_K_M.gguf` (3.1 GB, cached in `/kaggle/tmp/hf`, not saved with the notebook) and runs:

1. **gate**: the out-of-scope name `secure_compare` is masked, in-scope names are not
2. **trie**: with every attempt forced to fail, no retry replays a failed token sequence
3. **halt**: the run ends in `GRACEFUL_HALT` at `k = K_max = 3`

then one real repair attempt. Expect `SANDBOX CHECKS` with three `PASS`.

If you see *unknown model architecture*, the bundled llama.cpp predates Gemma 4: set a newer `LLAMA_CPP_VERSION` in Step 4 and rebuild.

In [ ]:
os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # a 3 GB model is fastest on a single T4
!python experiments/sandbox_smoke.py --n-gpu-layers -1

Optional: repeat with the larger E4B model (5.0 GB) to compare repair quality.

In [ ]:
# !python experiments/sandbox_smoke.py --n-gpu-layers -1 \
#     --gguf-repo unsloth/gemma-4-E4B-it-GGUF --gguf-file gemma-4-E4B-it-Q4_K_M.gguf

## Step 7 — Build validated SWE-bench pilot tasks
Kaggle has no Docker, so we only keep instances this environment can actually evaluate: each candidate is checked out, and kept only if its FAIL_TO_PASS tests **fail on the original code and pass with the official fix** (`--validate`).

Repos are cloned to `/kaggle/tmp/swe` (not persisted). In a new session, rerun this cell; the same `--seed` gives the same tasks.

Expect: `wrote N task(s)` and a per-repo keep/drop table. Instances dropped for environment reasons (old dependencies, compiled extensions) are normal.

On Kaggle's Python 3.13, older `requests`/`pytest` instances usually fail validation (their dependencies are too old), so the pilot tends to be mostly or entirely **sympy**. Say so when reporting results.

In [ ]:
PILOT_REPOS = ["sympy/sympy", "pallets/flask", "psf/requests", "pytest-dev/pytest"]
PILOT_N = 20
PY = sys.executable        # the notebook's own interpreter runs the repos' tests
!python experiments/prepare_tasks.py --repos {" ".join(PILOT_REPOS)} --validate --n {PILOT_N} \
    --max-candidates 80 --seed 0 --repos-dir /kaggle/tmp/swe --out /kaggle/working/tasks.jsonl \
    --test-timeout 300 --python {PY}

In [ ]:
import pandas as pd
val = pd.read_json("/kaggle/working/tasks_validation.jsonl", lines=True)
display(val.groupby("repo")["kept"].agg(kept="sum", tried="count"))
display(val[~val.kept][["instance_id", "base_exit", "gold_exit"]].head(10))

## Step 8 — Pilot ablation benchmark (resumable)
The model answers with **SEARCH/REPLACE edit blocks** (only the changed lines), applied to the real file before the reflex check and tests. Large files are shown as an issue-relevant excerpt; the excerpt is built the same way for every configuration.

Runs the five configurations bug by bug and appends results to `RESULTS_DIR` (`/kaggle/working/results_E4B`):

| Config | What it is |
|---|---|
| `baseline` | unconstrained, up to 10 turns, full transcript kept |
| `baseline_k3` | unconstrained, but the **same 3-turn cap and short context as H-NEG**; separates the method from just stopping early |
| `reflex_only` | + Loop 1 reflex check |
| `static_mask` | + identifier-scope logit masking |
| `full` | full H-NEG: CPG + mask + anti-replay guard + pushdown state machine + GRACEFUL_HALT |

- **Resumable:** rerun the same cell to continue. `--time-budget-hours` stops cleanly before Kaggle's 12-hour limit.
- **Provenance:** `run_meta.json` records the model's SHA-256, a code fingerprint, the task-set hash, the GPU and every setting. A resume with a different model, code or governance settings is refused, so one results folder always comes from one setup.
- **Unattended:** use **Save Version → Save & Run All (Commit)**.

Start with `LIMIT = 2` to check timing, then set `LIMIT = None` for all bugs.

**Same for every config:** the prompt with a worked edit example, the lenient edit parser (it also accepts unified diffs), fuzzy SEARCH matching, and the retry message showing the closest file lines. **Only `full`:** the anti-replay guard, which masks the one token that would complete a no-op edit, a failed (SEARCH, REPLACE) pair, or a SEARCH text that was already not found.

**Context window:** if the baseline's growing conversation outgrows the window partway through a bug, that bug counts as a **failure for that configuration** (`context_overflow`), not an error, because hitting the ceiling is the failure mode H-NEG targets. A bug whose *first* prompt doesn't fit is excluded from every configuration and listed in `errors.jsonl`.

**This notebook is set for the full run (`LIMIT = None`).** With Save & Run All, every cell reruns on a fresh machine: Step 4 installs the cached wheel, Step 7 re-validates the same tasks (same seed), and the cell below restores earlier results if you attached the previous version's output.

In [ ]:
# Resume across sessions: a new session / "Save & Run All" starts with an empty /kaggle/working.
# To continue an interrupted run, attach that notebook version's output (Add Input -> Your Work ->
# this notebook). This cell copies its results_*/traces_* folders back. Only the folder whose name
# matches the current model AND code fingerprint is used by Step 8; older ones are just kept.
import glob, os, shutil
for prev in sorted(glob.glob("/kaggle/input/**/results_*", recursive=True) + glob.glob("/kaggle/input/**/traces_*", recursive=True)):
    name = os.path.basename(prev)
    dst = f"/kaggle/working/{name}"
    if os.path.isdir(prev) and not os.path.exists(dst):
        shutil.copytree(prev, dst)
        print(f"restored {name}/ ({len(os.listdir(dst))} files)")
print("folders now in /kaggle/working:", sorted(d for d in os.listdir("/kaggle/working") if d.startswith(("results_", "traces_"))) or "none")


### Choose the benchmark model
**E4B is the model for results.** E2B produced no applicable edit in the first pilot.

Results go to `results_<model>_<code fingerprint>/`, e.g. `results_E4B_918612c5e74163c1/`. A new code version therefore always starts a fresh folder and never mixes with earlier runs, while an interrupted run of the *same* code resumes where it stopped (attach the previous version's output, see the cell above).


In [ ]:
from experiments.run_benchmark import _code_fingerprint
EXPECTED_FP = "918612c5e74163c1"          # fingerprint of the zip this notebook was set up for
CODE_FP = _code_fingerprint()
print("code fingerprint:", CODE_FP, "OK" if CODE_FP == EXPECTED_FP else f"<-- expected {EXPECTED_FP}: is the newest Dataset version attached?")

BENCH_MODEL = "E4B"            # "E4B" (results) or "E2B" (fast controller checks)
GGUF = {"E2B": ("unsloth/gemma-4-E2B-it-GGUF", "gemma-4-E2B-it-Q4_K_M.gguf"),
        "E4B": ("unsloth/gemma-4-E4B-it-GGUF", "gemma-4-E4B-it-Q4_K_M.gguf")}
GGUF_REPO, GGUF_FILE = GGUF[BENCH_MODEL]
RESULTS_DIR = f"/kaggle/working/results_{BENCH_MODEL}_{CODE_FP}"
TRACE_DIR = f"/kaggle/working/traces_{BENCH_MODEL}_{CODE_FP}"
N_CTX = 32768                  # lower to 16384 if the model fails to load (out of GPU memory)
done = sum(1 for _ in open(f"{RESULTS_DIR}/per_bug_full.csv")) - 1 if os.path.exists(f"{RESULTS_DIR}/per_bug_full.csv") else 0
print(BENCH_MODEL, GGUF_FILE, "->", RESULTS_DIR, f"(resuming: {done} bugs already done)" if done else "(fresh run)")


### Step 8a — Sanity check one real prompt (30 s)
Sends the first task's prompt to the model, exactly as the benchmark will, and shows what comes back. With Save & Run All it runs automatically; check its output in the log afterwards.

Expect: `output tokens` well above 0 (typically 50-400), `stop: eog <turn|>`, and a reply containing `<<<<<<< SEARCH` … `>>>>>>> REPLACE`. If output tokens are ~0 or the stop token isn't `<turn|>`, stop and send me this output.

In [ ]:
import json, sys, os
sys.argv = ["notebook"]
os.environ["HF_HOME"] = "/kaggle/tmp/hf"; os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from pathlib import Path
from experiments.run_benchmark import build_messages
from experiments.edit_format import parse_edit_blocks, apply_edits
from src.llamacpp_backend import GGUFRunner

probe_runner = GGUFRunner(repo_id=GGUF_REPO, filename=GGUF_FILE, n_ctx=N_CTX,
                          n_gpu_layers=-1, max_new_tokens=1024)     # first use downloads the model
task = json.loads(open("/kaggle/working/tasks.jsonl").readline())
source = (Path(task["repo_path"]) / task["target_file"]).read_text()
msgs = build_messages(task, source)
text, out_ids, n_in, dt = probe_runner.generate(msgs)
st = probe_runner.last_stop
blocks = parse_edit_blocks(text)
_, err = apply_edits(source, blocks)
print(f"task {task['instance_id']}  prompt tokens {n_in}  output tokens {len(out_ids)}  {dt:.1f}s")
print(f"stop: {st.get('reason')} {st.get('text')!r}   edit blocks: {len(blocks)}   apply: {err or 'OK'}")
print("--- reply (special tokens visible) ---")
print(st.get("raw_reply", "")[:1500])
del probe_runner                                       # free GPU memory before Step 8
import gc; gc.collect()


In [ ]:
# FULL RUN: all validated bugs, all five configurations (~7 min per bug on E4B -> ~2.5 h for 20 bugs).
# Resumable: if the session ends, attach this version's output and rerun; finished bugs are skipped.
LIMIT = None     # None = all bugs. Use 2 for a quick timing check.
limit_arg = f"--limit {LIMIT}" if LIMIT else ""
!python experiments/run_benchmark.py --backend gguf --n-gpu-layers -1 --n-ctx {N_CTX} \
    --gguf-repo {GGUF_REPO} --gguf-file {GGUF_FILE} \
    --tasks /kaggle/working/tasks.jsonl \
    --configs baseline baseline_k3 reflex_only static_mask full \
    --out-dir {RESULTS_DIR} --trace-dir {TRACE_DIR} \
    --max-new-tokens 1024 --patch-format edit --test-timeout 300 --time-budget-hours 10 \
    {limit_arg}


### Step 8b — Per-bug overview and what the model replied
One line per bug and configuration (resolved? attempts? why it failed), then the first `full` reply for every bug.


In [ ]:
import json, glob, os, collections
rows = collections.defaultdict(dict)
for f in sorted(glob.glob(f"{RESULTS_DIR}/attempts_*.jsonl")):
    cfg = os.path.basename(f)[9:-6]
    for line in open(f):
        r = json.loads(line)
        a = r["attempts"]
        ok = any(x["test_exit"] == 0 for x in a)
        why = "FIXED" if ok else (a[-1]["apply_error"] or a[-1]["reflex_error"] or f"tests exit {a[-1]['test_exit']}")
        rows[r["instance_id"]][cfg] = f"{'Y' if ok else '-'}{len(a):>2}"
        if cfg == "full":
            rows[r["instance_id"]]["_why"] = str(why)[:60]
cfgs = ["baseline", "baseline_k3", "reflex_only", "static_mask", "full"]
print(f"{'bug':22}" + "".join(f"{c:>13}" for c in cfgs) + "   full: last outcome   (Y = fixed, number = attempts)")
for iid in sorted(rows):
    print(f"{iid:22}" + "".join(f"{rows[iid].get(c, '?'):>13}" for c in cfgs) + "   " + rows[iid].get("_why", ""))
fixed = {c: sum(1 for i in rows if rows[i].get(c, "-").startswith("Y")) for c in cfgs}
print("\nfixed per config:", fixed)

print("\n--- first reply of `full` per bug ---")
for line in open(f"{RESULTS_DIR}/attempts_full.jsonl"):
    r = json.loads(line); a = r["attempts"][0]
    print(f"=== {r['instance_id']} | out={a['tokens_out']} stop={a['stop_reason']} | apply={a['apply_error']} | exit={a['test_exit']}")
    print(a["reply"][:600]); print()


## Step 9 — Results: ablation table, savings with 95% CIs, calibration
`savings_report.py` produces the paper's table straight from the per-bug results, using only bugs completed by **every** configuration:
- the ablation table, including tokens and time **per resolved bug**
- paired comparisons with 95% bootstrap confidence intervals:
  - `full vs baseline`: the headline saving
  - `full vs baseline_k3`: what masking, trie and state machine add beyond the retry cap
  - `baseline_k3 vs baseline`: what the cap alone buys

It writes `savings_report.md` and `savings.json` into the results folder, next to `run_meta.json`. Copy numbers into the paper only from these files.

In [ ]:
# Optional ROI: set your own prices (inputs, not measurements)
GPU_COST_PER_HOUR = 0.35     # e.g. a cloud T4; use your real price
BUGS_PER_MONTH = 1000
!python experiments/savings_report.py --results-dir {RESULTS_DIR} \
    --gpu-cost-per-hour {GPU_COST_PER_HOUR} --bugs-per-month {BUGS_PER_MONTH}
import os
if os.path.exists(f"{RESULTS_DIR}/errors.jsonl"):
    print("Skipped bugs (not in the table):")
    print(open(f"{RESULTS_DIR}/errors.jsonl").read()[:2000])

In [ ]:
!python -m src.state_machine {TRACE_DIR}


## Step 9b — Claim verification
`verify_claims.py` checks every claim of the paper against this run and marks it **VERIFIED**, **NOT SUPPORTED** or **INCONCLUSIVE**:

- **Guarantees (C1-C5)**: loop elimination, the hard K_max bound, never re-submitting a failed patch, 0% syntax failures, no out-of-scope names. These are properties of the mechanism, so they can be checked at any n.
- **Effects (C6-C10)**: token reduction, speedup, governance beyond the retry cap (`full vs baseline_k3`, exact McNemar test), Pass@1 and cost per resolved bug. These need resolved bugs, and they hold for the model that ran, not for the 31B model in Table 6.3.

Writes `claims_verification.md` and `claims.json` next to `savings_report.md`. These two files plus `run_meta.json` are the evidence to cite.


In [ ]:
!python experiments/verify_claims.py --results-dir {RESULTS_DIR} --gpu-cost-per-hour {GPU_COST_PER_HOUR}


## Keep in mind
- These are **sandbox-model** numbers (E4B / E2B) on a validated subset; the `model` field and `run_meta.json` record which model produced them. They are not the paper's 31B results.
- Guarantees (C1-C5) carry over to any model because they come from the controller. Effect sizes (C6-C10) do not; report them as "measured on Gemma 4 E4B, n = …".
- Bugs whose file doesn't fit the context window are logged to `errors.jsonl` and left out of the summary.
- To save outputs: **Save Version**. Everything in `/kaggle/working` (results, traces, tasks, wheel) is kept.
